# train resnet50 on colab
Hafsa Rashid (23K-0064), Syeda Sara Ali (23K-0070)

## check gpu and mount drive

In [7]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

GPU 0: Tesla T4 (UUID: GPU-92964f00-998a-763c-8b90-d109078d0572)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
PROJECT = '/content/drive/MyDrive/CV A2 UPAR'
RUN_DIR = PROJECT + '/runs/resnet50'   # on drive so its safe if colab disconnects
os.makedirs(RUN_DIR, exist_ok=True)
print(os.listdir(PROJECT))

['metrics.py', 'model.py', 'dataset.py', 'config.py', 'make_submission.py', 'check_submission.py', 'test_model.py', 'train.py', 'evaluate.py', 'colab_train.ipynb', 'README.md', 'submission_code', 'runs']


## copy code to local disk

In [9]:
!rm -rf /content/code && mkdir -p /content/code
!cd "{PROJECT}" && cp *.py requirements.txt /content/code/ && cp -r submission_code sample_images data /content/code/
!ls /content/code

cp: cannot stat 'requirements.txt': No such file or directory
check_submission.py  dataset.py   make_submission.py  model.py	     train.py
config.py	     evaluate.py  metrics.py	      test_model.py


## download UPAR images

In [12]:
%cd /content/UPAR-Challenge-2027
import shutil, os
from pathlib import Path

# Market1501
src, dst = Path('data/Market-1501-v15.09.15'), Path('data/Market1501')
if src.exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    shutil.rmtree(src)
print('Market1501 ok:', dst.exists())

# other 2 datasets
import download_datasets as d
if not Path('data/PA100k/release_data').exists():
    d.prepare_pa100k(Path('data'))
if not Path('data/PETA/images').exists() or not any(Path('data/PETA/images').iterdir()):
    d.prepare_peta(Path('data'))
print('done')

/content/UPAR-Challenge-2027
Market1501 ok: True
done


In [13]:
# check that images are there
import pandas as pd
DATA = '/content/UPAR-Challenge-2027/data'
TRAIN_CSV = DATA + '/annotations/task1/train/gt.csv'
VAL_CSV = DATA + '/annotations/task1/val/gt.csv'
for name, csv in [('train', TRAIN_CSV), ('val', VAL_CSV)]:
    paths = pd.read_csv(csv).iloc[:, 0]
    missing = sum(not os.path.exists(os.path.join(DATA, p)) for p in paths)
    print(name, len(paths), 'images, missing:', missing)

train 97669 images, missing: 0
val 33407 images, missing: 0


## train resnet50 (3 epochs)

In [14]:
%cd /content/code
!python train.py --train_csv "{TRAIN_CSV}" --val_csv "{VAL_CSV}" --img_root "{DATA}" \
    --backbone resnet50 --epochs 3 --bs 64 --lr 3e-4 --workers 2 --light_aug --out "{RUN_DIR}"

/content/code
train: 97669  val: 33407  attributes: 40  device: cuda
epoch 1/3: 100% 1526/1526 [05:51<00:00,  4.34it/s]
{'mA': 80.30565388997395, 'F1': 82.34921773274739, 'score': 81.31329091389973, 'epoch': 1, 'loss': 0.30207089380621127, 'time_sec': 407}
epoch 2/3: 100% 1526/1526 [05:59<00:00,  4.24it/s]
{'mA': 84.35467783610027, 'F1': 85.3819808959961, 'score': 84.8649164835612, 'epoch': 2, 'loss': 0.19536390759021552, 'time_sec': 415}
epoch 3/3: 100% 1526/1526 [06:01<00:00,  4.22it/s]
{'mA': 84.58086903889973, 'F1': 86.23770141601562, 'score': 85.40101369222005, 'epoch': 3, 'loss': 0.15970173119756528, 'time_sec': 417}
val @0.5 : {'mA': 84.7335713704427, 'F1': 86.47328440348308, 'score': 85.59447987874348}
val tuned: {'mA': 87.21411641438802, 'F1': 85.96061197916667, 'score': 86.58273061116536}


In [17]:
import json
print(json.dumps(json.load(open(RUN_DIR + '/final.json')), indent=1))
for row in json.load(open(RUN_DIR + '/log.json')):
    print(row)

{
 "backbone": "resnet50",
 "epochs": 3,
 "batch_size": 64,
 "lr": 0.0003,
 "light_aug": true,
 "base": {
  "mA": 84.7335713704427,
  "F1": 86.47328440348308,
  "score": 85.59447987874348
 },
 "tuned": {
  "mA": 87.21411641438802,
  "F1": 85.96061197916667,
  "score": 86.58273061116536
 }
}
{'mA': 80.30565388997395, 'F1': 82.34921773274739, 'score': 81.31329091389973, 'epoch': 1, 'loss': 0.30207089380621127, 'time_sec': 407}
{'mA': 84.35467783610027, 'F1': 85.3819808959961, 'score': 84.8649164835612, 'epoch': 2, 'loss': 0.19536390759021552, 'time_sec': 415}
{'mA': 84.58086903889973, 'F1': 86.23770141601562, 'score': 85.40101369222005, 'epoch': 3, 'loss': 0.15970173119756528, 'time_sec': 417}


In [19]:
import os, shutil
from PIL import Image
R = '/content/UPAR-Challenge-2027'
os.makedirs('/content/code/data', exist_ok=True)
shutil.copy(R + '/data/annotations/task1/val/gt.csv', '/content/code/data/val_gt.csv')
shutil.copy(R + '/examples/task1/sample_code_submission/assets/attribute_prior.json', '/content/code/data/')

# sample images for check_submission
os.makedirs('/content/code/sample_images', exist_ok=True)
paths = pd.read_csv(VAL_CSV).iloc[:3, 0]
for i, p in enumerate(paths, 1):
    out = f'/content/code/sample_images/ped{i}.png'
    if not os.path.exists(out):
        Image.open(os.path.join(DATA, p)).convert('RGB').save(out)

print('data:', os.listdir('/content/code/data'))
print('run folder:', os.listdir(RUN_DIR))

data: ['val_gt.csv', 'attribute_prior.json']
run folder: ['best.pt', 'log.json', 'last.pt', 'val_probs.npy', 'thresholds.json', 'final.json', 'results']


## evaluate and make submission zip

In [20]:
import os, shutil
from PIL import Image
import pandas as pd
R = '/content/UPAR-Challenge-2027'
os.makedirs('/content/code/data', exist_ok=True)
os.makedirs('/content/code/sample_images', exist_ok=True)
shutil.copy(R + '/data/annotations/task1/val/gt.csv', '/content/code/data/val_gt.csv')
shutil.copy(R + '/examples/task1/sample_code_submission/assets/attribute_prior.json', '/content/code/data/')
for i, p in enumerate(pd.read_csv(VAL_CSV).iloc[:3, 0], 1):
    out = f'/content/code/sample_images/ped{i}.png'
    if not os.path.exists(out):
        Image.open(os.path.join(DATA, p)).convert('RGB').save(out)
print('data:', os.listdir('/content/code/data'))

%cd /content/code
!python evaluate.py --probs "{RUN_DIR}/val_probs.npy" --thr "{RUN_DIR}/thresholds.json" --out "{RUN_DIR}/results"
!python make_submission.py --model "{RUN_DIR}/best.pt" --backbone resnet50 --thr_json "{RUN_DIR}/thresholds.json" --out "{RUN_DIR}/resnet50_tuned.zip"
!python check_submission.py "{RUN_DIR}/resnet50_tuned.zip"

data: ['val_gt.csv', 'attribute_prior.json']
/content/code
val images: 33407, attributes: 40
images per dataset: {'Market1501': 16458, 'PA100k': 9986, 'PETA': 6963}

                method    dataset    mA    F1  score
       Prior baseline Market1501 57.50 58.87  58.17
       Prior baseline     PA100k 50.00 57.02  53.28
       Prior baseline       PETA 50.00 52.88  51.40
       Prior baseline       MEAN 52.50 56.25  54.28
       Our model @0.5 Market1501 85.70 87.23  86.46
       Our model @0.5     PA100k 83.28 85.23  84.24
       Our model @0.5       PETA 85.22 86.96  86.08
       Our model @0.5       MEAN 84.73 86.47  85.59
Our model + tuned thr Market1501 87.61 86.59  87.10
Our model + tuned thr     PA100k 86.37 85.08  85.72
Our model + tuned thr       PETA 87.67 86.21  86.93
Our model + tuned thr       MEAN 87.21 85.96  86.58

                   n_attr    mA    F1
group                               
Accessory            5.0  79.2  61.9
Age                  3.0  82.3  68.0
Hair   

In [23]:
!mkdir -p /content/code/submission_code

In [24]:
%%writefile /content/code/submission_code/run.py
# run.py - our code submission for UPAR 2027 Track 1
# codabench imports this file and calls load_model(), predict_image(), predict_batch()
# shift logits so prob >= tuned thr gives output > 0.5
import io, json, os
import numpy as np
import torch
import torch.nn as nn
import torchvision
from PIL import Image

HERE = os.path.dirname(os.path.abspath(__file__))
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
BATCH_SIZE = 64
model = None
device = "cpu"
info = {}
logit_shift = None
IMAGE_KEYS = ["image", "img", "pil_image", "image_path", "path", "file", "filename", "image_array", "array", "data"]


def load_model():
    global model, device, info, logit_shift
    info = json.load(open(os.path.join(HERE, "assets", "meta.json")))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_attr = len(info["attributes"])
    backbone = info.get("backbone", "convnext_small")
    net = getattr(torchvision.models, backbone)(weights=None)
    if backbone.startswith("resnet"):
        net.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(net.fc.in_features, n_attr))
    else:
        net.classifier[-1] = nn.Sequential(nn.Dropout(0.0), nn.Linear(net.classifier[-1].in_features, n_attr))
    state = torch.load(os.path.join(HERE, "assets", "model.pt"), map_location="cpu", weights_only=True)
    net.load_state_dict({k: v.float() for k, v in state.items()})
    model = net.to(device).eval()
    thr = np.clip(np.array(info["thr"], dtype=np.float32), 1e-3, 1 - 1e-3)
    logit_shift = torch.tensor(np.log(thr / (1 - thr)), device=device)


def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
    if isinstance(x, (bytes, bytearray)):
        return Image.open(io.BytesIO(x)).convert("RGB")
    if isinstance(x, (str, os.PathLike)):
        return Image.open(x).convert("RGB")
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.ndim == 3 and x.shape[0] in (1, 3) and x.shape[-1] not in (1, 3):
        x = x.transpose(1, 2, 0)
    if x.dtype != np.uint8:
        if x.max() <= 1.0:
            x = x * 255
        x = x.clip(0, 255).astype(np.uint8)
    return Image.fromarray(x).convert("RGB")


def get_image(sample):
    if not isinstance(sample, dict):
        return to_pil(sample)
    for k in IMAGE_KEYS:
        if sample.get(k) is not None:
            return to_pil(sample[k])
    for k, v in sample.items():
        if k == "attribute_names":
            continue
        try:
            return to_pil(v)
        except Exception:
            pass
    raise KeyError(f"could not find image in sample, keys: {list(sample.keys())}")


def predict_batch(samples):
    if model is None:
        load_model()
    h, w = info["h"], info["w"]
    imgs = [np.asarray(get_image(s).resize((w, h), Image.BILINEAR), dtype=np.uint8) for s in samples]
    x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float() / 255
    x = (x - MEAN) / STD
    outputs = []
    with torch.no_grad():
        for i in range(0, len(x), BATCH_SIZE):
            xb = x[i:i + BATCH_SIZE].to(device)
            p = (torch.sigmoid(model(xb)) + torch.sigmoid(model(torch.flip(xb, dims=[3])))) / 2
            p = p.clamp(1e-6, 1 - 1e-6)
            p = torch.sigmoid(torch.log(p / (1 - p)) - logit_shift)
            outputs.append(p.cpu())
    probs = torch.cat(outputs).numpy()
    our_names = info["attributes"]
    if samples and isinstance(samples[0], dict) and samples[0].get("attribute_names"):
        cols = []
        for name in samples[0]["attribute_names"]:
            if name in our_names:
                cols.append(probs[:, our_names.index(name)])
            else:
                cols.append(np.zeros(len(probs), dtype=np.float32))
        probs = np.stack(cols, axis=1)
    probs = np.nan_to_num(probs, nan=0.0).clip(0.0, 1.0)
    return [[float(v) for v in row] for row in probs]


def predict_image(sample):
    return predict_batch([sample])[0]

Writing /content/code/submission_code/run.py


In [25]:
!cp /content/UPAR-Challenge-2027/examples/task1/sample_code_submission/metadata.yaml /content/code/submission_code/
!ls /content/code/submission_code

metadata.yaml  run.py


In [26]:
%cd /content/code
!python make_submission.py --model "{RUN_DIR}/best.pt" --backbone resnet50 --thr_json "{RUN_DIR}/thresholds.json" --out "{RUN_DIR}/resnet50_tuned.zip"
!python check_submission.py "{RUN_DIR}/resnet50_tuned.zip"

/content/code
saved /content/drive/MyDrive/CV A2 UPAR/runs/resnet50/resnet50_tuned.zip (43.8 MB)
[PASS] zip has run.py, metadata.yaml, assets/ at root  (['run.py', 'metadata.yaml', 'assets/model.pt', 'assets/meta.json'])
[PASS] zip size  (43.8 MB)
[PASS] run.py has load_model / predict_image / predict_batch
[PASS] load_model() works offline
[PASS] predict_image gives 40 floats
[PASS] all values between 0 and 1  (min 0.001, max 0.999)
[PASS] predict_batch gives one row per image
[PASS] predict_batch same as predict_image
[PASS] shuffled attribute order is followed
[PASS] input type: PIL image
[PASS] input type: numpy array
[PASS] input type: tensor CHW
[PASS] input type: bytes
[PASS] input type: file path
[PASS] same yes/no decisions as our model  (0 different)

ALL TESTS PASSED
